# Write Oka 2026 OD600 data as linked ATST

Run from `read_shared_data/`. Three source tables become three readouts in one study. Excluded curves remain present and annotated. The S5 orphan value is assigned to the final blank reading as a documented inference.


In [1]:
from pathlib import Path
import csv
import datetime
import pandas as pd
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    ReadoutIds,
    Study,
)

root = Path.cwd()
sources = {
    "MOI_controlled": "TableS5_growth curve MOI controlled.csv",
    "Plaque_derived": "TableS6_growth curve from plaque.csv",
    "Sewage_isolates": "TableS7_growth curve from sewage.csv",
}
file_info = FileMetadata(
    data={
        "file_name": "atst_oka_2026.atst.txt",
        "format": "ATST",
        "format_version": "0.1",
        "created_on": datetime.date.today().isoformat(),
        "field_delimiter": "TAB",
        "encoding": "UTF-8",
    }
)
study = Study(
    data={
        "title": "Feature-Based Growth Curve Classification Enables Efficient Phage Discrimination",
        "study_id": "doi:10.3390/v18010092",
        "authors": "Yuma Oka; Keidai Miyakawa; Moe Yamazaki; Yuki Maruyama",
        "publication_doi": "10.3390/v18010092",
        "publication_date": "2026",
        "description": "Three published OD600 source tables: controlled MOI, plaque-derived T-phages, and sewage phages.",
    }
)
entities = Entities(
    tables={
        name: EntityTable(
            name="ENTITIES",
            table_name=name,
            pk=pk,
            data=pd.read_csv(
                root / "entities_definitions" / f"{name}.csv",
                dtype=str,
                keep_default_na=False,
            ),
        )
        for name, pk in [
            ("ISOLATES", "ISOLATE_ID"),
            ("PHAGES", "PHAGE_ID"),
            ("MEDIA", "MEDIUM_ID"),
        ]
    }
)


def curve_id(index):
    letters = ""
    while index:
        index, remainder = divmod(index - 1, 26)
        letters = chr(65 + remainder) + letters
    return "curve_" + letters


readouts = {}
for readout_id, source_name in sources.items():
    with (root / "sources" / source_name).open(
        encoding="utf-8-sig", newline=""
    ) as file:
        rows = list(csv.reader(file))
    controlled = readout_id == "MOI_controlled"
    header = rows[0] if controlled else rows[1]
    status = [""] * len(header) if controlled else rows[0]
    data = rows[1:] if controlled else rows[2:]
    if controlled:
        # S5 row 140 contains only 0.2358 in source column 65 and no time.
        # The final timed row (row 139) lacks only its column-68 blank reading.
        assert (
            len(data[-1]) == 68
            and data[-1][0] == ""
            and data[-1][64] == "0.2358"
            and sum(bool(v) for v in data[-1]) == 1
        )
        assert data[-2][0] == "82199.99" and data[-2][67] == ""
        data[-2][67] = data[-1][64]
        data = data[:-1]
    assert all(row[0] for row in data)
    readings = pd.DataFrame({"Time": [float(row[0]) for row in data]})
    layout_rows = []
    counts = {}
    for column, label in enumerate(header[1:], 1):
        counts[label] = counts.get(label, 0) + 1
        cid = curve_id(column)
        readings[cid] = [float(row[column]) if row[column] else None for row in data]
        blank = label == "blank"
        control = label == "no_phage"
        if controlled:
            phage_id = "" if blank or control else label.split("_MOI")[0]
            moi = (
                ""
                if blank or control
                else {"1": "1", "_01": "0.1", "_001": "0.01"}[label.split("_MOI", 1)[1]]
            )
            isolate = "" if blank else "E_coli_NBRC13168"
            condition = label
            species = ""
        elif readout_id == "Plaque_derived":
            phage_id = "" if blank or control else label
            moi = ""
            isolate = "" if blank else "E_coli_NBRC13168"
            condition = label
            species = ""
        else:
            phage_id = "" if blank or control else f"sewage_{label}"
            moi = ""
            isolate = "" if blank else "E_coli_NBRC13898"
            condition = label
            species = status[column] if status[column] != "excluded" else ""
        layout_rows.append(
            {
                "curve_id": cid,
                "type": "blank"
                if blank
                else "uninfected_control"
                if control
                else "phage_exposed",
                "isolate_id": isolate,
                "phage_id": phage_id,
                "moi": moi,
                "replicate": str(counts[label]),
            }
        )
    assay = Assay(
        data={
            "readout_type": "absorbance",
            "readout_unit": "od600",
            "time_unit": "s",
            "medium_id": "LB",
            "culture_type": "planktonic",
            "temperature_c": "37",
            "instrument": "Varioskan LUX microplate reader (Thermo Fisher Scientific)",
            "plate": "96-well plate",
            "shaking": "continuous orbital shaking, 600 rpm",
            "conditions": "Mid-exponential OD600 0.5 culture diluted 10-fold in fresh LB; T-phages at MOI 1, 0.1 or 0.01"
            if controlled
            else "1 µL freshly collected plaque filtrate added to 99 µL bacterial culture without phage titer determination"
            if readout_id == "Plaque_derived"
            else "Same growth-curve protocol as plaque-derived experiment; environmental phages from sewage",
            "source_reference": "doi:10.3390/v18010092",
        }
    )
    metadata = Metadata(
        data={
            "source_file": "read_shared_data/sources/" + source_name,
            "source_header_rows": "1" if controlled else "1-2",
            "source_data_rows": "2:139" if controlled else "3:147",
            "source_columns": f"A:{'BP' if controlled else 'Z' if readout_id == 'Plaque_derived' else 'AA'}",
            "source_exclusion": "None"
            if controlled
            else "S6 source column Q (T6 replicate 1) marked excluded from downstream analysis"
            if readout_id == "Plaque_derived"
            else "S7 source columns C, Q, V (isolate 2, 16, 21) marked excluded from downstream analysis",
            "source_alignment_repair": "S5 row 140 column BM (source field 65) 0.2358 assigned to missing blank at row 139 column BP (field 68), time 82199.99 s; inferred from sole orphan and adjacent blank readings"
            if controlled
            else "none",
        }
    )
    readouts[readout_id] = ATSTFile(
        file_info=file_info,
        study=study,
        metadata=metadata,
        assay=assay,
        entities=entities,
        layout=Layout(data=pd.DataFrame(layout_rows)),
        readings=Readings(data=readings),
    )
readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": list(readouts)}))
for item in readouts.values():
    item.readout_ids = readout_ids
bundle = MultiReadoutATST(file_info, study, readout_ids, readouts, entities)
validate_atst(bundle)
display(
    pd.DataFrame(
        [
            {
                "readout_id": key,
                "curves": len(item.layout.data),
                "time_points": len(item.readings.data),
            }
            for key, item in readouts.items()
        ]
    )
)


        readout_id  curves  time_points
0   MOI_controlled      67          138
1   Plaque_derived      25          145
2  Sewage_isolates      26          145


In [2]:
output = write_multi_readout_atst(
    bundle, root.parent / "atst_oka_2026", linked_files=True
)
print(output)


/Users/juanmantilla/Documents/TKI/ATST/examples/Public data /Oka 2026/atst_oka_2026/atst_oka_2026.atst.txt
